### Bag of Stories
**Description:** Compute Jensen-Shannon divergence in a set of stories.

**Contributor:** Florentina Armaselu  

In [167]:
# Import required packages
from pathlib import Path
import numpy as np
from scipy.spatial.distance import jensenshannon
from collections import Counter
import datetime as dt
import os
import math
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

Define the input, output paths.

In [168]:
# Define the base directory relative to the current script location
base_dir = os.getcwd()
# Set the path to the input data folder for whole stories
input = os.path.join(base_dir, "./../data/grimm/")
# Set the path to the input data folder for segmented stories with TextTiling
input_seg_tt = os.path.join(base_dir, "./../data/annotated_stories/grimm/text-tiling/")
# Set the path to the input data folder for segmented stories with LLM episodes
input_seg_llm = os.path.join(base_dir, "./../data/annotated_stories/grimm/llm-episodes/")
# Set the path to the output folder for JSD results on the input stories
output = os.path.join(base_dir, "./../results/jsd/grimm/")

Define constants.

In [169]:
# String constants for JSD types
JSD_ADJACENT = 'adjacent'
JSD_CONTEXTUAL = 'contextual'

# String constants for segmentation methods
SEG_EQUAL_LENGTH = 'equal-length'
SEG_TEXT_TILING = 'text-tiling'
SEG_LLM_EPISODE = 'llm-episode'

# String constants for annotation elements
ANN_SEG = 'Segment' # Label for segments
ANN_SEG_PAIR = 'Segment Pair' # Label for segment pairs
ANN_SEP = '-'*40 # Separator for segments

# String constants for output subfolders and file formats
OUT_CSV = 'csv'
OUT_PNG = 'png'

# String constants for file naming
FN_ADJACENT = 'adj'
FN_CONTEXTUAL = 'ctx'
FN_EQUAL_LENGTH = 'el'
FN_TEXT_TILING = 'tt'
FN_LLM_EPISODE = 'llmep'

# String constant for JSD labels
JSD_LABEL = 'JSD'
JSD_AVG = 'JSD Avg'
JSD_STD = 'JSD Std'
JSD_DEV = 'JSD Dev'

Define functions for story segmentation.

In [170]:
# Divide stories into segments of approximately equal length in number of tokens
def el_segment_story(story, seg_length=100):
    """
    Segments a story into smaller parts of approximately equal length in tokens.
    
    Args:
        story (str): The full text of the story.
        segment_length (int): The desired length of each segment in tokens.
        
    Returns:
        list: A list of segments, each containing approximately `segment_length` tokens.
    """   
    # Split the story into tokens
    tokens = story.split()
    
    # Calculate the number of segments based on the desired segment length
    seg_count = math.ceil(len(tokens) / seg_length)
    
    # Length of each segment in tokens more equally distributed
    distrib_length = math.ceil(len(tokens) / seg_count) 
    
    # Create segments of approximately equal length
    # This will ensure that the segments are more evenly distributed in terms of token count
    segments = []
    for i in range(0, len(tokens), distrib_length):
        segment_tokens = tokens[i:i + distrib_length]
        segment = ' '.join(segment_tokens)
        if segment.strip():  # Add only non-empty segments
            segments.append(segment.strip())

    return segments

In [171]:
# Read an annotated story and return the segments.
def ann_segment_story(story):
    """
    Reads a text-tiling segmented story from a string and returns the segments.
    
    Args:
        story (str): The full text of the story.
        
    Returns:
        list: A list of segments extracted from the story.
    """
    segments = []
    # Split the story into parts based on the ANN_SEG label and ANN_SEP separator
    parts = story.split(ANN_SEP)
    for part in parts:
        # Skip the first line if it contains the ANN_SEG label
        if part.strip().startswith(ANN_SEG):
            part = '\n'.join(part.split('\n')[2:])  # Remove the line containing the annotation label
        if part.strip():  # Add only non-empty segments
            segments.append(part.strip())
    
    return segments

Define functions for computing Jensen-Shannon divergence in stories.

In [172]:
# Compute Jensen-Shannon divergence for two segments
def compute_jsd(segment1, segment2):
    """
    Computes the Jensen-Shannon divergence between two segments.
    
    Args:
        segment1 (str): The first segment of text.
        segment2 (str): The second segment of text.
        
    Returns:
        float: The Jensen-Shannon divergence between the two segments.
    """
    # Tokenize the segments
    tokens1 = segment1.split()
    tokens2 = segment2.split()
    
    # Count the frequency of each token in both segments
    freq_list1 = Counter(tokens1)
    freq_list2 = Counter(tokens2)
    
    # Create a combined set of all unique tokens
    all_tokens = set(freq_list1.keys()).union(set(freq_list2.keys()))
    
    # Create probability distributions for both segments
    prob1 = np.array([freq_list1[token] / len(tokens1) if len(tokens1) > 0 else 0 for token in all_tokens])
    prob2 = np.array([freq_list2[token] / len(tokens2) if len(tokens2) > 0 else 0 for token in all_tokens])
    
    # Compute the Jensen-Shannon divergence
    jsd = jensenshannon(prob1, prob2, base=2) ** 2
    
    return jsd

In [ ]:
# Compute Jensen-Shannon divergence for segments in a story
def compute_jsd_for_story(story, jsd_type='adjacent', seg_method='equal-length'):
    """
    Computes the Jensen-Shannon divergence for all segments in a story.
    
    Args:
        story (str): The full text of the story.
        jsd_type (str): The type of JSD to compute ('adjacent' or 'contextual').
        seg_method (str): The method used for segmenting the stories ('equal-length', 'text-tiling', 'llm-episodes').
        
    Returns:
        list: A list of Jensen-Shannon divergence values for each pair of adjacent segments.
    """
    # Segment the story into smaller parts
    segments = ann_segment_story(story) if (seg_method == SEG_TEXT_TILING or seg_method == SEG_LLM_EPISODE) else \
    el_segment_story(story, seg_length=100) 

    # Compute JSD for each pair of adjacent or contextual segments
    jsd_values = []
    for i in range(len(segments) - 1):
        if jsd_type == JSD_CONTEXTUAL:
            # For contextual JSD, compute JSD between the next segment and all previous segments
            jsd = compute_jsd(' '.join(segments[0:i+1]), segments[i + 1])
            # Uncomment the debugging line below to check the segments. See examples in the segment_check folder
            # print(f"{ANN_SEG} 0-{i}: {' '.join(segments[0:i+1])}\n{ANN_SEG} {i+1}: {segments[i + 1]}\n{ANN_SEP}")
        else:
            # For adjacent JSD, compute JSD between the current segment and the next one  
            jsd = compute_jsd(segments[i], segments[i + 1])
            # Uncomment the debugging line below to check the segments. See examples in the segment_check folder
            # print(f"{ANN_SEG} {i}: {segments[i]}\n{ANN_SEG} {i+1}: {segments[i + 1]}\n{ANN_SEP}")
        jsd_values.append(jsd)
    
    return jsd_values

Define the function for analysing the stories and saving the results.

In [179]:
# Process a set of stories to compute JSD values for story segments and save results as CSV and PNG files
def process_jsd_stories(jsd_type, seg_method):
    """
    Processes all stories in the input directory and saves the JSD results to the output directory.
    
    Args:
        jsd_type (str): The type of JSD to compute ('adjacent' or 'contextual').
        seg_method (str): The method used for segmenting the stories ('equal-length', 'text-tiling', 'llm-episodes').
    """
    cnt_processed = 0 # Counter for processed files
    jsd_avg_values = [] # List to store JSD averages for each story
    story_names = [] # List of story names
    num_chars = [] # Counter list for the number of characters processed
    num_toks = [] # Counter list for the number of tokens processed
    num_segs = [] # Counter list for the number of segments processed
    fn_jsd_type=FN_ADJACENT # File name suffix for JSD type, default is 'adjacent'
    fn_seg_meth=FN_EQUAL_LENGTH # File name suffix for segmentation method, default is 'equal-length'
    inp = input # Default input path for whole stories

    # Select the input path and the file name suffix based on the segmentation method
    if seg_method == SEG_TEXT_TILING:
        inp = input_seg_tt
        fn_seg_meth = FN_TEXT_TILING
    elif seg_method == SEG_LLM_EPISODE:
        inp = input_seg_llm
        fn_seg_meth = FN_LLM_EPISODE

    # Select the file name suffix based on the JSD type
    if jsd_type == JSD_CONTEXTUAL:
        fn_jsd_type = FN_CONTEXTUAL

    # Read the story files sequentially from the input folder.
    for file_path in Path(inp).glob("*.txt"):
        with open(file_path, "r", encoding="utf-8") as input_file:
            story = input_file.read()
    
            # Print the name of the file being processed.
            print(f'Processing file: {file_path.name}')

            if (seg_method == SEG_TEXT_TILING or seg_method == SEG_LLM_EPISODE):
                # Get the title of the story
                start = story.find(ANN_SEG + " 0:\n") + len(ANN_SEG + " 0:\n")
                end = story.find(ANN_SEP)
                story_name = story[start:end].strip()
                # Get the body of the story, excluding the title
                body = story[end+len(ANN_SEP):]
            else:   
                # Get the title of the story
                story_name = story.split('\n')[0]
                # Get the body of the story, excluding the title
                body = story[story.find('\n')+1:]
                
            # Compute JSD values for the story
            jsd_values = compute_jsd_for_story(body, jsd_type, seg_method)

            # Create a timestamp for the output files
            timestamp = dt.datetime.now().strftime('%Y%m%d_%H%M%S')

            jsd_avg = np.mean(jsd_values)  # JSD average for the story
            jsd_avg_values.append(jsd_avg)  # Store the JSD average for the story
            story_names.append(story_name)
            num_chars.append(len(body))  # Store the number of characters in the story
            num_toks.append(len(body.split()))  # Store the number of tokens in the story
            num_segs.append(len(jsd_values)+1)  # Store the number of segments in the story

            # Story JSD table
            df = pd.DataFrame({
                ANN_SEG_PAIR : range(1, len(jsd_values) + 1),
                JSD_LABEL : jsd_values,
                JSD_DEV : [jsd_val - jsd_avg for jsd_val in jsd_values],  # JSD deviation from the average by segment pair
                JSD_AVG : jsd_avg, # JSD average for the story
                JSD_STD : np.std(jsd_values)  # Standard deviation of JSD for the story
            })
            
            # Save the story DataFrame to a CSV file
            csv_path = os.path.join(output, OUT_CSV, f"{file_path.stem}_{fn_jsd_type}_{fn_seg_meth}_{JSD_LABEL.lower()}_{timestamp}.{OUT_CSV}")
            df.to_csv(csv_path, index=False)

            # Save the line plot as PNG
            plt.figure(figsize=(8, 4))
            plt.plot(df[ANN_SEG_PAIR], df[JSD_LABEL], marker='o')
            plt.xlabel(ANN_SEG_PAIR)
            plt.ylabel('Jensen-Shannon Divergence')
            plt.title(f'Surprisal (JSD) across {jsd_type} {seg_method.replace('llm', 'LLM')} segments: {story_name}')
            plt.grid(True)
            plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(integer=True))  # Ensure integer ticks
            plt.xlim(left=0)  # Start x-axis from 0
            # Set x-ticks to include the first and last segment numbers
            ticks = np.unique(np.concatenate(([df[ANN_SEG_PAIR].min()], plt.gca().get_xticks(), [df[ANN_SEG_PAIR].max()])))
            plt.xticks(ticks.astype(int))  # Set ticks (as integers)
            plt.tight_layout()
            png_path = os.path.join(output, OUT_PNG, f"{file_path.stem}_{fn_jsd_type}_{fn_seg_meth}_{JSD_LABEL.lower()}_{timestamp}.{OUT_PNG}")
            plt.savefig(png_path)
            plt.close()

            cnt_processed += 1

    jsd_avg_by_run = np.mean(jsd_avg_values)  # Average JSD accross the run

    # Summary DataFrame for the run
    df_avg = pd.DataFrame({
        'ID': range(1, len(jsd_avg_values) + 1),
        'Story': story_names,
        'Characters': num_chars,  # Number of characters by story
        'Tokens': num_toks,  # Number of tokens by story  
        'Segments': num_segs,  # Number of segments by story
        JSD_AVG: jsd_avg_values,  # JSD average by story
        JSD_DEV: [jsd_val - jsd_avg_by_run for jsd_val in jsd_avg_values],  # JSD deviation from the average by story
        JSD_AVG + ' Run': jsd_avg_by_run, # JSD average by run
        JSD_STD + ' Run': np.std(jsd_avg_values)  # Standard deviation of JSD by run
    })

    # Save the summary DataFrame to a CSV file
    csv_avg_path = os.path.join(output, OUT_CSV, f"summary_{fn_jsd_type}_{fn_seg_meth}_{JSD_LABEL.lower()}_{timestamp}.{OUT_CSV}")
    df_avg.to_csv(csv_avg_path, index=False)
    
    # Print the number of files saved in the output folder and the summary file name.
    print(f'End processing, {cnt_processed}, files saved to the {Path(output).name} folder.')
    print(f'Summary of JSD values across the run saved to the {Path(csv_avg_path).name} file.')

Validate input, read the stories, analyse them, and save the results.  

In [ ]:
# Choose the JSD type and segmentation method
jsd_type = JSD_CONTEXTUAL # JSD_ADJACENT or JSD_CONTEXTUAL
seg_method = SEG_TEXT_TILING  # SEG_EQUAL_LENGTH or SEG_TEXT_TILING or SEG_LLM_EPISODE

# Validate the JSD type
if jsd_type not in [JSD_ADJACENT, JSD_CONTEXTUAL]:
    raise ValueError("Invalid JSD type. Choose from ", JSD_ADJACENT, "or ", JSD_CONTEXTUAL, ".")

# Validate the segmentation method and set the input path accordingly
if seg_method not in [SEG_EQUAL_LENGTH, SEG_TEXT_TILING, SEG_LLM_EPISODE]:
    raise ValueError("Invalid segmentation method. Choose from ", SEG_EQUAL_LENGTH, ", ", SEG_TEXT_TILING, "or ", SEG_LLM_EPISODE, ".")

# Process the stories with the specified JSD type and segmentation method
process_jsd_stories(jsd_type, seg_method)